In [4]:
import os
import requests
import pandas as pd
import numpy as np
import time
from io import StringIO
from tqdm import tqdm

ALPHA_VANTAGE_KEY = '60G1A3ENUDKBDCO1'
LISTING_URL = f"https://www.alphavantage.co/query?function=LISTING_STATUS&apikey={ALPHA_VANTAGE_KEY}"
BASE_URL = "https://www.alphavantage.co/query"
OUT_DIR = "data"

os.makedirs(OUT_DIR, exist_ok=True)

def calc_zscore_signal(df, window=20, threshold=1.0):
    df['mean'] = df['close'].rolling(window).mean()
    df['std'] = df['close'].rolling(window).std()
    df['zscore'] = (df['close'] - df['mean']) / df['std']
    df['zscore_signal'] = 0
    df.loc[df['zscore'] > threshold, 'zscore_signal'] = -1
    df.loc[df['zscore'] < -threshold, 'zscore_signal'] = 1
    return df['zscore_signal']

def calc_sma_signal(df, fast=20, slow=50):
    df['sma_fast'] = df['close'].rolling(fast).mean()
    df['sma_slow'] = df['close'].rolling(slow).mean()
    cross = df['sma_fast'] - df['sma_slow']
    df['sma_signal'] = 0
    df.loc[cross > 0, 'sma_signal'] = 1
    df.loc[cross < 0, 'sma_signal'] = -1
    df['sma_signal'] = df['sma_signal'].diff().fillna(0)
    return df['sma_signal']

def calc_stoch_signal(df, k_period=14, d_period=3, overbought=80, oversold=20):
    low_min = df['low'].rolling(window=k_period).min()
    high_max = df['high'].rolling(window=k_period).max()
    df['%K'] = 100 * (df['close'] - low_min) / (high_max - low_min)
    df['%D'] = df['%K'].rolling(window=d_period).mean()
    df['stoch_signal'] = 0
    buy = (df['%K'] < oversold) & (df['%K'] > df['%D'].shift(1))
    sell = (df['%K'] > overbought) & (df['%K'] < df['%D'].shift(1))
    df.loc[buy, 'stoch_signal'] = 1
    df.loc[sell, 'stoch_signal'] = -1
    return df['stoch_signal']

def fetch_listing():
    print('Downloading ticker listing...')
    resp = requests.get(LISTING_URL)
    if resp.status_code != 200:
        raise Exception(f"Failed to fetch listing: {resp.status_code}")
    tickers_df = pd.read_csv(StringIO(resp.text))
    tickers = tickers_df[tickers_df['status'] == 'Active']['symbol'].tolist()
    print(f"Found {len(tickers)} active tickers.")
    return tickers

def fetch_daily(ticker):
    params = {
        'function': 'TIME_SERIES_DAILY_ADJUSTED',
        'symbol': ticker,
        'outputsize': 'full',
        'apikey': ALPHA_VANTAGE_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    ts = data.get('Time Series (Daily)', {})
    if not ts:
        return None
    df = pd.DataFrame(ts).T
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.rename(columns={
        '1. open': 'open',
        '2. high': 'high',
        '3. low': 'low',
        '4. close': 'close',
        '5. adjusted close': 'adj_close',
        '6. volume': 'volume'
    })
    for col in ['open', 'high', 'low', 'close', 'adj_close', 'volume']:
        df[col] = df[col].astype(float)
    df = df.tail(252)
    return df

def save_json(ticker, df):
    out_path = os.path.join(OUT_DIR, f"{ticker}.json")
    df_out = df[['close', 'zscore_signal', 'sma_signal', 'stoch_signal']].copy()
    df_out['date'] = df_out.index
    # Use ISO date string for compatibility
    df_out['date'] = df_out['date'].dt.strftime('%Y-%m-%d')
    df_out.reset_index(drop=True, inplace=True)
    df_out.to_json(out_path, orient='records', date_format='iso')
    
def main():
    tickers = fetch_listing()
    # ONLY tickers starting at or after 'P'
    tickers = [t for t in tickers if isinstance(t, str) and t and t[0].upper() >= 'P']
    print(f"Tickers to process (starting with 'P' and after): {len(tickers)}")

    batch_size = 75  # Use 75 for premium API key, or lower for free
    for i in tqdm(range(0, len(tickers), batch_size)):
        batch = tickers[i:i+batch_size]
        for ticker in batch:
            json_path = os.path.join(OUT_DIR, f"{ticker}.json")
            if os.path.exists(json_path):
                print(f"{ticker} already processed. Skipping.")
                continue
            try:
                df = fetch_daily(ticker)
                if df is None or df.empty:
                    print(f"Skipping {ticker} (no data).")
                    continue
                df['zscore_signal'] = calc_zscore_signal(df)
                df['sma_signal'] = calc_sma_signal(df)
                df['stoch_signal'] = calc_stoch_signal(df)
                save_json(ticker, df)
                print(f"{ticker} processed and saved.")
            except Exception as e:
                print(f"Error for {ticker}: {e}")
        # Sleep to respect the premium rate limit (75 calls/min)
        if i + batch_size < len(tickers):
            time.sleep(61)
    print("Done.")



if __name__ == '__main__':
    main()


Found 12029 active tickers.
Tickers to process (starting with 'P' and after): 3937


  0%|                                                                                           | 0/53 [00:00<?, ?it/s]

PAA already processed. Skipping.
PAAA already processed. Skipping.
PAAS already processed. Skipping.
PAB already processed. Skipping.
PABD already processed. Skipping.
PABU already processed. Skipping.
PAC already processed. Skipping.
PACB already processed. Skipping.
PACHU already processed. Skipping.
PACI-U already processed. Skipping.
PACK already processed. Skipping.
PACS already processed. Skipping.
PACWP already processed. Skipping.
PACX already processed. Skipping.
PACXW already processed. Skipping.
PAG already processed. Skipping.
PAGP already processed. Skipping.
PAGS already processed. Skipping.
PAHC already processed. Skipping.
PAI already processed. Skipping.
PAII-U already processed. Skipping.
PAL already processed. Skipping.
PALC already processed. Skipping.
PALD already processed. Skipping.
PALI already processed. Skipping.
PALL already processed. Skipping.
PALU already processed. Skipping.
PAM already processed. Skipping.
PAMC already processed. Skipping.
PAMT already p

  2%|█▌                                                                                 | 1/53 [01:01<52:54, 61.05s/it]

PBJL already processed. Skipping.
PBJN already processed. Skipping.
PBL already processed. Skipping.
PBM already processed. Skipping.
PBMR already processed. Skipping.
PBMWW already processed. Skipping.
PBMY already processed. Skipping.
PBNV already processed. Skipping.
PBOC already processed. Skipping.
PBP already processed. Skipping.
PBPB already processed. Skipping.
PBR already processed. Skipping.
PBR-A already processed. Skipping.
PBSE already processed. Skipping.
PBT already processed. Skipping.
PBTP already processed. Skipping.
PBUS already processed. Skipping.
PBW already processed. Skipping.
PBYI already processed. Skipping.
PC already processed. Skipping.
PCAP already processed. Skipping.
PCAPU already processed. Skipping.
PCAPW already processed. Skipping.
PCAR already processed. Skipping.
PCB already processed. Skipping.
PCCE already processed. Skipping.
PCCT already processed. Skipping.
PCCTU already processed. Skipping.
PCEF already processed. Skipping.
PCEM already proce

...

YORKW processed and saved.
YORW processed and saved.
YOSH processed and saved.
YOU processed and saved.
YOUL processed and saved.
YPF processed and saved.
YQ processed and saved.
YQQQ processed and saved.
YRD processed and saved.
YSEP processed and saved.
YSG processed and saved.
YSPY processed and saved.
YSXT processed and saved.
YTRA processed and saved.
YUM processed and saved.
YUMC processed and saved.
YXI processed and saved.
YXT processed and saved.
YYAI processed and saved.
YYGH processed and saved.
YYY processed and saved.
Z processed and saved.
ZALT processed and saved.
ZAP processed and saved.
ZAUG processed and saved.
ZAZZT processed and saved.
ZBAI processed and saved.
ZBAO processed and saved.
ZBH processed and saved.
ZBIO processed and saved.
ZBRA processed and saved.
ZBZX processed and saved.
ZBZZT processed and saved.
ZCMD processed and saved.
ZCZZT processed and saved.
ZD processed and saved.
ZDAI processed and saved.
ZDEK processed and saved.
ZDGE processed and saved.

 98%|█████████████████████████████████████████████████████████████████████████████▌ | 52/53 [1:32:48<01:56, 116.75s/it]

ZOCT processed and saved.
ZONE processed and saved.
ZOOZ processed and saved.
ZOOZW processed and saved.
ZROZ processed and saved.
ZS processed and saved.
ZSB processed and saved.
ZSC processed and saved.
ZSEP processed and saved.
ZSL processed and saved.
ZSPC processed and saved.
ZTAQU processed and saved.
ZTAX processed and saved.
ZTEK processed and saved.
ZTEST processed and saved.
ZTO processed and saved.
ZTR processed and saved.
ZTS processed and saved.
ZTST processed and saved.
ZUMZ processed and saved.
ZURA processed and saved.
ZVIA processed and saved.
ZVOL processed and saved.
ZVRA processed and saved.
ZVSA processed and saved.
ZVV processed and saved.
ZVZZT processed and saved.
ZWS processed and saved.
ZWZZT processed and saved.
ZXYZ-A processed and saved.
ZXZZT processed and saved.
ZYBT processed and saved.
ZYME processed and saved.
ZYNE processed and saved.
ZYXI processed and saved.
ZZK processed and saved.


100%|███████████████████████████████████████████████████████████████████████████████| 53/53 [1:33:16<00:00, 105.59s/it]

ZZZ processed and saved.
Done.
